In [1]:
!pip install "protobuf<7"

In [2]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [3]:
from langchain_core.documents import Document

In [4]:
#Text data
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("data/python.txt", encoding="utf-8")

C:\Users\Saket\AppData\Local\Temp\ipykernel_29752\910593988.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [5]:
document = loader.load()

In [6]:
document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [7]:
#pdf data

In [8]:
from langchain_community.document_loaders.pdf import PyPDFLoader
pdf_loader = PyPDFLoader("data/pdfs/research2.pdf")
document = pdf_loader.load()

## Data Ingestion

In [9]:
#data => documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))

    return all_docs

In [10]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


## chunks

In [11]:
!pip install langchain_text_splitters

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap= chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs
    

In [13]:
chunks =split_docs(all_pdf_documents)

In [14]:
len(chunks)

321

## Embedding

In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-V2"):
        self.model_name= model_name
        print("loading model ...", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())
    # to generate embedding
    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape:", embeddings.shape)
        return embeddings

In [17]:
embedding_manager = EmbeddingManager()

loading model ... all-MiniLM-L6-V2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


C:\Users\Saket\AppData\Local\Temp\ipykernel_29752\4049928347.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


## Vector Store

In [18]:
import chromadb
import uuid

In [19]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection =None
        self.client=None
        self._initialize_store()
    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        #create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collection
        self.collection =self.client.get_or_create_collection(
            name =self.collection_name,
            metadata={"description":"vector store collection for pdf embeddings in RAG"}
        )
        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())
        
    #create a function to add all the documents
    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")
        #store => ids, embedding, document, metadata
        ids =[]
        all_metadata =[]
        documents_content =[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            # metadata of individual metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] =i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            # now document 
            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )
        print("total documents added in vector store=",len(documents_content))
        print("docs in colections:", self.collection.count())
                
            

In [20]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [21]:
# chunks=> embedding
texts =[doc.page_content for doc in chunks]
embedding = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape: (321, 384)
total documents added in vector store= 321
docs in colections: 321


## Retrival pipelines implementation

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager =embedding_manager
        self.vector_store= vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        #query => embedding
        query_embeddings= self.embedding_manager.generate_embeddings([query])[0]

        #sementic search
        results =self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results=top_k
        )
        #cosine similarity
        retrieved_docs=[]
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]
            for i, (doc_id, metadata, documents, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1- distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "documents": metadata,
                        "distance": distance,
                        "similaity_score":similarity_score,
                        "rank" :i+1
                        

                    })
            print(f"retrieved {len(retrieved_docs)} documents")
        else:
            print("no documents found")
        return retrieved_docs
                
        
        

In [24]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [25]:
rag_retriever.retrieve("what is RAG")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_1099ae3b-a761-46d7-89dc-c097e1ecd67f',
  'documents': {'creationdate': '2024-03-28T00:54:45+00:00',
   'moddate': '2024-03-28T00:54:45+00:00',
   'doc_index': 88,
   'author': '',
   'subject': '',
   'page': 0,
   'page_label': '1',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'content_length': 288,
   'total_pages': 21,
   'source': 'data/pdfs\\research2.pdf',
   'keywords': '',
   'trapped': '/False',
   'title': '',
   'producer': 'pdfTeX-1.40.25',
   'creator': 'LaTeX with hyperref'},
  'distance': 0.42267584800720215,
  'similaity_score': 0.5773241519927979,
  'rank': 1},
 {'id': 'doc_b8548545-c09a-4865-af80-f1a9bfc10334',
  'documents': {'subject': '',
   'content_length': 461,
   'doc_index': 91,
   'page': 1,
   'author': '',
   'creator': 'LaTeX with hyperref',
   'keywords': '',
   'page_label': '2',
   'total_pages': 21,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (